In [4]:
import tensorflow as tf
import os
import numpy as np

# ۱. بارگذاری مدلی که در مرحله قبل آموزش دادیم
model = tf.keras.models.load_model('human_detection_base.keras')

# ۲. پیکربندی مبدل TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# ۳. ساخت تابع دیتاست نماینده برای کالیبراسیون دقیق
DATA_DIR = '../data/human-and-non-human/training_set'
IMG_SIZE = 96

def representative_data_gen():
    dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=1,
        color_mode="grayscale",
        shuffle=True,
        seed=42 # ثابت کردن سید برای اطمینان از مخلوط شدن مساوی عکس‌های هر دو کلاس
    )
    for images, labels in dataset.take(100):
        # باید حتماً به نامپای و نوع float32 تبدیل شود تا مبدل گیج نشود
        processed_image = (images.numpy() / 255.0).astype(np.float32)
        yield [processed_image]

# ۴. اعمال تنظیمات کوانتیزاسیون ۸-بیتی
converter.representative_dataset = representative_data_gen
# اجبار مبدل به استفاده از عملیات‌های ۸-بیتی (مناسب برای میکروکنترلرها)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# تنظیم ورودی و خروجی شبکه به فرمت int8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("در حال تبدیل و فشرده‌سازی مدل... (این کار ممکن است چند ثانیه طول بکشد)")
tflite_quant_model = converter.convert()

در حال تبدیل و فشرده‌سازی مدل... (این کار ممکن است چند ثانیه طول بکشد)
INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpf3pm08jd\assets


INFO:tensorflow:Assets written to: C:\Users\User\AppData\Local\Temp\tmpf3pm08jd\assets


Saved artifact at 'C:\Users\User\AppData\Local\Temp\tmpf3pm08jd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 96, 96, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2795135667664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135669584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135669008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135669968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135669776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135670352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135670160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135670736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135669200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2795135671696: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\User\anaconda3\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Found 8017 files belonging to 2 classes.


In [2]:
# ذخیره فایل TFLite
tflite_model_path = 'model.tflite'
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)

# محاسبه و چاپ حجم مدل‌ها
base_model_size = os.path.getsize('human_detection_base.keras') / 1024
tflite_model_size = os.path.getsize(tflite_model_path) / 1024

print(f"حجم مدل اولیه: {base_model_size:.2f} KB")
print(f"حجم مدل کوانتیزه‌شده: {tflite_model_size:.2f} KB")
print(f"میزان فشرده‌سازی: {(base_model_size / tflite_model_size):.1f} برابر کوچکتر!")

حجم مدل اولیه: 520.60 KB
حجم مدل کوانتیزه‌شده: 46.63 KB
میزان فشرده‌سازی: 11.2 برابر کوچکتر!
